# Lab 05 - Block Ciphers, Padding Modes, and Error Propagation

## **Yusupov A.**  

---

### References
* UPJŠ Lectures & Lab 05 Materials (`KRS2026-Lab05`, doc. RNDr. Jozef Jirásek, PhD.)
* Python Cryptodome Documentation (`pycryptodomex` / `Crypto.Cipher` / `Crypto.Util.Padding`)
* Usage of AI: Assistance with code structure, error analysis across block cipher modes, and AEAD modern modes via Gemini LLM.
* Communication with other students: No reference.

## 0. Universal Environment Setup & Imports
We provide robust imports supporting either `Cryptodome` (from `pycryptodomex`) or `Crypto` (from `pycryptodome`), ensuring seamless execution in any JupyterLab / Conda environment.

In [6]:
!pip install pycryptodome

In [7]:
try:
    from Cryptodome.Cipher import DES, AES
    from Cryptodome.Random import get_random_bytes
    from Cryptodome.Util.Padding import pad, unpad
    from Cryptodome.Util import Counter
    print("Successfully imported from Cryptodome (pycryptodomex)!")
except ImportError:
    from Crypto.Cipher import DES, AES
    from Crypto.Random import get_random_bytes
    from Crypto.Util.Padding import pad, unpad
    from Crypto.Util import Counter
    print("Successfully imported from Crypto (pycryptodome)!")

from base64 import b64encode, b64decode

# Helper function to display hex blocks formatted cleanly
def wrap(text, size=8):
    for i in range((len(text) + size - 1) // size):
        chunk = text[i * size : (i + 1) * size]
        hex_part = " ".join(f"{c:02X}" for c in chunk)
        ascii_part = "".join(chr(c) if 32 <= c < 127 else "." for c in chunk)
        print(f"  Block {i+1} [{hex_part:<23}]  |  {ascii_part}")

Successfully imported from Cryptodome (pycryptodomex)!


---
## Task 1: Encipher text not multiple of 8 with different padding modes

When using block ciphers like DES (block size 8 bytes), plaintexts whose length is not a multiple of 8 must be padded.
We test three standard padding modes provided by `Crypto.Util.Padding`:
1. **PKCS#7 (`pkcs7`)**: Appends $N$ bytes, each with value $N$.
2. **ISO 7816-4 (`iso7816`)**: Appends byte `0x80` followed by zeroes.
3. **ANSI X.923 (`x923`)**: Appends zero bytes with the last byte indicating pad length.

In [8]:
data_short = b'Short'  # 5 bytes (not a multiple of 8)
key = get_random_bytes(8)
padding_modes = ['pkcs7', 'iso7816', 'x923']

print(f"Original plaintext: {data_short} (length: {len(data_short)} bytes)\n")

for p_style in padding_modes:
    # 1. Pad
    padded = pad(data_short, 8, style=p_style)

    # 2. Encrypt
    cipher = DES.new(key, DES.MODE_ECB)
    ct = cipher.encrypt(padded)

    # 3. Decrypt & Unpad
    decipher = DES.new(key, DES.MODE_ECB)
    decrypted_padded = decipher.decrypt(ct)
    unpadded = unpad(decrypted_padded, 8, style=p_style)

    print(f"=== Mode: {p_style.upper()} ===")
    print(f"  Padded bytes : {padded}")
    print(f"  Padded hex   : {padded.hex()}")
    print(f"  Ciphertext   : {ct.hex()}")
    print(f"  Recovered    : {unpadded.decode('utf-8')}\n")

Original plaintext: b'Short' (length: 5 bytes)

=== Mode: PKCS7 ===
  Padded bytes : b'Short\x03\x03\x03'
  Padded hex   : 53686f7274030303
  Ciphertext   : 77ce8b00f679b959
  Recovered    : Short

=== Mode: ISO7816 ===
  Padded bytes : b'Short\x80\x00\x00'
  Padded hex   : 53686f7274800000
  Ciphertext   : df3cccda85dcc065
  Recovered    : Short

=== Mode: X923 ===
  Padded bytes : b'Short\x00\x00\x03'
  Padded hex   : 53686f7274000003
  Ciphertext   : f685b2e955221f7e
  Recovered    : Short



---
## Task 2: Block Cipher Modes & 1-Bit Error Propagation

**Requirements:**
* Use at least 3 blocks (here 3 blocks of 8 bytes = 24 bytes: `TestUPJS12345678TestUPJS`).
* Encrypt using the same key across modes: **ECB, CBC, OFB, CFB, CTR, OPENPGP**.
* Invert **one bit** in the second block of the ciphertext (byte index 11, `ct[11] ^ 64`).
* Decrypt the corrupted ciphertext and observe error propagation differences across modes.

In [9]:
# 3 blocks of 8 bytes = 24 bytes
plaintext_3blocks = b'TestUPJS12345678TestUPJS'
key_t2 = get_random_bytes(8)
iv_t2 = get_random_bytes(8)
nonce_t2 = get_random_bytes(4)

print("Plaintext (3 blocks):")
wrap(plaintext_3blocks)
print("-" * 60)

# Function to flip 1 bit (bit 6 of byte index)
def flip_bit(buf, byte_idx=11, mask=0x40):
    b = bytearray(buf)
    b[byte_idx] ^= mask
    return bytes(b)

modes_config = {
    'ECB': {
        'enc': lambda: DES.new(key_t2, DES.MODE_ECB),
        'dec': lambda iv_arg: DES.new(key_t2, DES.MODE_ECB),
        'is_openpgp': False
    },
    'CBC': {
        'enc': lambda: DES.new(key_t2, DES.MODE_CBC, iv=iv_t2),
        'dec': lambda iv_arg: DES.new(key_t2, DES.MODE_CBC, iv=iv_arg),
        'is_openpgp': False
    },
    'OFB': {
        'enc': lambda: DES.new(key_t2, DES.MODE_OFB, iv=iv_t2),
        'dec': lambda iv_arg: DES.new(key_t2, DES.MODE_OFB, iv=iv_arg),
        'is_openpgp': False
    },
    'CFB': {
        'enc': lambda: DES.new(key_t2, DES.MODE_CFB, iv=iv_t2, segment_size=64),
        'dec': lambda iv_arg: DES.new(key_t2, DES.MODE_CFB, iv=iv_arg, segment_size=64),
        'is_openpgp': False
    },
    'CTR': {
        'enc': lambda: DES.new(key_t2, DES.MODE_CTR, counter=Counter.new(32, prefix=nonce_t2)),
        'dec': lambda iv_arg: DES.new(key_t2, DES.MODE_CTR, counter=Counter.new(32, prefix=nonce_t2)),
        'is_openpgp': False
    },
    'OPENPGP': {
        'enc': lambda: DES.new(key_t2, DES.MODE_OPENPGP, iv=iv_t2),
        'dec': lambda iv_arg: DES.new(key_t2, DES.MODE_OPENPGP, iv=iv_arg),
        'is_openpgp': True
    }
}

for m_name, m_cfg in modes_config.items():
    # Encrypt
    c_enc = m_cfg['enc']()
    ct = c_enc.encrypt(plaintext_3blocks)

    if m_cfg['is_openpgp']:
        # OpenPGP mode prepends 10 bytes (8-byte IV + 2-byte repeat checksum)
        # Byte 11 of plaintext corresponds to ciphertext byte index 10 + 11 = 21
        ct_corrupted = flip_bit(ct, byte_idx=21, mask=0x40)
        c_dec = m_cfg['dec'](ct_corrupted[:10])
        pt_corrupted = c_dec.decrypt(ct_corrupted[10:])
        display_ct = ct[10:]
    else:
        # Standard modes: byte index 11 is in Block 2 (indices 8..15)
        ct_corrupted = flip_bit(ct, byte_idx=11, mask=0x40)
        c_dec = m_cfg['dec'](iv_t2)
        pt_corrupted = c_dec.decrypt(ct_corrupted)
        display_ct = ct

    print(f"\n>>> MODE: {m_name} <<<")
    print("Ciphertext:")
    wrap(display_ct)
    print("Decrypted plaintext with corrupted bit in Block 2 (byte 11):")
    wrap(pt_corrupted)

Plaintext (3 blocks):
  Block 1 [54 65 73 74 55 50 4A 53]  |  TestUPJS
  Block 2 [31 32 33 34 35 36 37 38]  |  12345678
  Block 3 [54 65 73 74 55 50 4A 53]  |  TestUPJS
------------------------------------------------------------

>>> MODE: ECB <<<
Ciphertext:
  Block 1 [04 E5 B0 01 0D 0A C0 58]  |  .......X
  Block 2 [B1 76 F7 34 91 AE BB 2B]  |  .v.4...+
  Block 3 [04 E5 B0 01 0D 0A C0 58]  |  .......X
Decrypted plaintext with corrupted bit in Block 2 (byte 11):
  Block 1 [54 65 73 74 55 50 4A 53]  |  TestUPJS
  Block 2 [F5 A3 88 B3 E3 F2 29 E4]  |  ......).
  Block 3 [54 65 73 74 55 50 4A 53]  |  TestUPJS

>>> MODE: CBC <<<
Ciphertext:
  Block 1 [A4 6E 60 09 08 8A F4 69]  |  .n`....i
  Block 2 [26 F4 9F 1C 47 C9 54 7E]  |  &...G.T~
  Block 3 [CC A0 23 A5 28 E7 0B 49]  |  ..#.(..I
Decrypted plaintext with corrupted bit in Block 2 (byte 11):
  Block 1 [54 65 73 74 55 50 4A 53]  |  TestUPJS
  Block 2 [D3 5D 22 C8 D0 69 17 1C]  |  .]"..i..
  Block 3 [54 65 73 34 55 50 4A 53]  |  Tes4UPJ

### Analysis of Error Propagation Differences

| Mode | Block 1 | Block 2 (Corrupted Bit) | Block 3 | Theoretical Explanation |
|:---|:---|:---|:---|:---|
| **ECB** | Correct | **Completely Garbled** | Correct | $P_i = D_K(C_i)$. Blocks are deciphered independently; only the block containing the corrupted bit is randomized. |
| **CBC** | Correct | **Completely Garbled** | **1 bit flipped** | $P_2 = D_K(C_2) \oplus C_1$ (garbles block 2). But $P_3 = D_K(C_3) \oplus C_2$, so the flipped bit in $C_2$ directly flips the exact matching bit in $P_3$. |
| **OFB** | Correct | **1 bit flipped** | Correct | Keystream $O_i$ is generated independently of ciphertext: $P_i = C_i \oplus O_i$. Bit errors do not propagate to any other bits or blocks. |
| **CFB** | Correct | **1 bit flipped** | **Completely Garbled** | $P_2 = C_2 \oplus E_K(C_1)$ (flips 1 bit). Then $C_2$ acts as cipher input for block 3: $P_3 = C_3 \oplus E_K(C_2)$, totally randomizing block 3. |
| **CTR** | Correct | **1 bit flipped** | Correct | Stream mode where keystream is derived from counter values: $P_i = C_i \oplus E_K(\text{Counter}_i)$. Zero error propagation. |
| **OPENPGP** | Correct | **Completely Garbled** | **1 bit flipped** | Variant of CFB with prefix resynchronization, mirroring CBC/CFB block dependency across adjacent blocks. |

---
## Task 3 (Bonus*): Modern Modes of Operation for AES

Modern modes of operation provide **Authenticated Encryption with Associated Data (AEAD)**:
* **Confidentiality:** Plaintext is encrypted.
* **Integrity & Authenticity:** An authentication tag (MAC) verifies that neither ciphertext nor metadata (AD) was tampered with.

We evaluate **AES-GCM, AES-CCM, AES-EAX, and AES-SIV** and verify how tampering with even 1 bit in ciphertext causes the decryption to safely reject the data.

In [10]:
payload = b"Confidential payload requiring authenticated encryption!"
auth_header = b"TransactionID: 98765-KRS2026"
aes_key_16 = get_random_bytes(16)  # AES-128

modern_modes = [
    ("GCM", AES.MODE_GCM),
    ("EAX", AES.MODE_EAX),
    ("CCM", AES.MODE_CCM),
    ("SIV", AES.MODE_SIV)
]

for m_name, m_mode in modern_modes:
    # SIV requires a 32-byte key (16 bytes for auth + 16 bytes for enc)
    cur_key = get_random_bytes(32) if m_name == "SIV" else aes_key_16

    # 1. Encryption & Tag Generation
    if m_name == "CCM":
        c = AES.new(cur_key, m_mode, mac_len=16)
        c.update(auth_header)
        ciphertext = c.encrypt(payload)
        tag = c.digest()
        nonce_val = c.nonce
    elif m_name == "SIV":
        c = AES.new(cur_key, m_mode)
        c.update(auth_header)
        ciphertext, tag = c.encrypt_and_digest(payload)
        nonce_val = None
    else:
        c = AES.new(cur_key, m_mode)
        c.update(auth_header)
        ciphertext, tag = c.encrypt_and_digest(payload)
        nonce_val = c.nonce

    print(f"=== AES-{m_name} (Authenticated Encryption) ===")
    print(f"  Ciphertext (hex) : {ciphertext.hex()[:40]}...")
    print(f"  Auth Tag (hex)   : {tag.hex()}")

    # 2. Tampering test: Invert 1 bit in ciphertext
    tampered_ct = bytearray(ciphertext)
    tampered_ct[0] ^= 0x01

    # 3. Decryption & Verification
    if m_name == "CCM":
        dec = AES.new(cur_key, m_mode, nonce=nonce_val, mac_len=16)
    elif m_name == "SIV":
        dec = AES.new(cur_key, m_mode)
    else:
        dec = AES.new(cur_key, m_mode, nonce=nonce_val)

    dec.update(auth_header)
    try:
        dec.decrypt_and_verify(bytes(tampered_ct), tag)
        print("  Status: Decrypted successfully.")
    except ValueError:
        print("  Status: MAC verification failed! Tampering was detected and rejected.\n")

=== AES-GCM (Authenticated Encryption) ===
  Ciphertext (hex) : d17b844d907ab5319a7bec787da3d6af96ad500d...
  Auth Tag (hex)   : 36387c361bf8019561f466243d6cb5e0
  Status: MAC verification failed! Tampering was detected and rejected.

=== AES-EAX (Authenticated Encryption) ===
  Ciphertext (hex) : 50732de7fca9cbd797a33a80f62041252478b06b...
  Auth Tag (hex)   : 95e8587f16d2d59ebad1d1b872ce3160
  Status: MAC verification failed! Tampering was detected and rejected.

=== AES-CCM (Authenticated Encryption) ===
  Ciphertext (hex) : 7c1cb7ace9d69aac15628a2051278f4e6df1a9e8...
  Auth Tag (hex)   : 286bb1bd0b24807497c0a5d2e5d04441
  Status: MAC verification failed! Tampering was detected and rejected.

=== AES-SIV (Authenticated Encryption) ===
  Ciphertext (hex) : 43287bb895a4f274586dad90c9d72ad088f810ae...
  Auth Tag (hex)   : f91af3438dc6f26354205617b2ea736b
  Status: MAC verification failed! Tampering was detected and rejected.

